# Visium HD - SCA019

In [1]:
# TODO: Visium HD reader in harpy currently does not work due to bug fix in spatialdata-io (https://github.com/scverse/spatialdata-io/pull/303). Update harpy reader.
# TODO: Discuss option with Arne to overwrite tissue positions with our own registered coordinates.
# TODO: Rebin data based on table layer
# TODO: BMK reader, Nova-ST reader, Curio reader, StereoSeq reader

## Import packages

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json
import gc

from dask_image.imread import imread
import spatialdata_plot

from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [4]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue
from _visium_hd import visium_hd

## Create sdata

In [ ]:
dataset_id = "SCA019"

path_outs = f'D:/2024-07-UTBenchmark-SpC/data/raw/Visium_HD/4/COUNTS/{dataset_id}/{dataset_id}_reseq/outs'
path_img = f'D:/2024-07-UTBenchmark-SpC/data/raw/Visium_HD/4/IMAGES/{dataset_id}_HE_hires.tif'
output_path = f'D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/{dataset_id}/harpy/sdata.zarr'

tissue_annotation_file = f'D:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/{dataset_id}/tissue_annotation/tissue.geojson'

In [6]:
sdata = visium_hd(
    path=path_outs, 
    dataset_id=dataset_id,
    filtered_counts_file=False,
    bin_size=[2, 8, 16, 32], 
    bins_as_squares=True, 
    fullres_image_file=path_img,
    load_all_images=True,
    output=output_path
)
sdata

AssertionError: Cannot locate the feature slice file, please ensure the file is present in the D:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA019\SpaceRanger\outs directory and/or adjustthe `dataset_id` parameter

In [ ]:
# # Check spots
# sdata.pl.render_images(
#     element=f"{dataset_id}_full_image"
# ).pl.render_labels(
#     element=f"{dataset_id}_square_002um_labels", fill_alpha=0.2, contour_px=1, outline_alpha=0, color='in_tissue'
# ).pl.show(
#     coordinate_systems=dataset_id,
#     colorbar=False,
#     dpi=600
# )

## Calculate pixel size

In [ ]:
# Determine pixel size in microns
import tifffile

with tifffile.TiffFile(path_img) as tif:
    xres = tif.pages[0].tags["XResolution"].value  # (numerator, denominator)
    yres = tif.pages[0].tags["YResolution"].value
    unit = tif.pages[0].tags["ResolutionUnit"].value

    px_per_cm_x = xres[0] / xres[1]
    px_per_cm_y = yres[0] / yres[1]

    if unit.name == "CENTIMETER":
        microns_per_pixel_x = 10_000 / px_per_cm_x
        microns_per_pixel_y = 10_000 / px_per_cm_y
    elif unit.name == "INCH":
        microns_per_pixel_x = 25_400 / px_per_cm_x
        microns_per_pixel_y = 25_400 / px_per_cm_y
    else:
        raise ValueError(f"Unknown ResolutionUnit: {unit}")

if microns_per_pixel_x == microns_per_pixel_y:
    microns_per_pixel = microns_per_pixel_x
    print(f"Pixel size: {microns_per_pixel:.3f} µm")
else:
    print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")


Pixel size: 0.442 µm


## Create bounding box for total capture area

In [ ]:
# Add bounding box for total capture area
import shapely

geoms = sdata.shapes[f'{dataset_id}_square_002um'].geometry
bbox = shapely.MultiPolygon(list(geoms)).minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_square_002um'], get_all=True),
    overwrite = True,
)

In [ ]:
# # Check capture area bbox
# sdata.pl.render_images(
#     element=f"{dataset_id}_full_image"
# ).pl.render_shapes(
#     element=f'{dataset_id}_square_002um', color='black', fill_alpha=0.1, outline_width=0, outline_alpha=1, outline_color='black'
# ).pl.render_shapes(
#     element=f'{dataset_id}_capture_area', color='#1f77b4', fill_alpha=0.1, outline_width=1.5, outline_alpha=1, outline_color='red'
# ).pl.show(
#     coordinate_systems=dataset_id,
#     colorbar=False,
#     pad_extent=1000,
#     dpi=600
# )

In [ ]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

Area of capture area: 44.89 mm²


In [ ]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

Dimensions of capture area bbox: 0.67 cm x 0.67 cm


## Add tissue annotation

In [ ]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_square_002um'], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2025-08-14 11:37:20,638 - harpy.image._manager - INFO - Writing results to layer 'SCA010_tissue_mask'


In [ ]:
# # Check tissue annotation
# sdata.pl.render_images(
#     element=f"{dataset_id}_full_image"
# ).pl.render_shapes(
#     element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=1, outline_alpha=1, outline_color='black'
# ).pl.show(
#     coordinate_systems=dataset_id,
#     colorbar=False,
#     dpi=300
# )

In [ ]:
# Determine which spots are in tissue annotation
sdata = add_in_tissue(
    sdata, 
    table_layer = 'square_002um', 
    tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
)

2025-08-14 11:40:49,925 - harpy.utils._io - WARNING - layer with name 'square_002um' already exists. Overwriting...


In [ ]:
# # Check spots in tissue
# sdata.pl.render_images(
#     element=f"{dataset_id}_full_image"
# ).pl.render_labels(
#     element=f"{dataset_id}_square_002um_labels", fill_alpha=0.2, contour_px=1, outline_alpha=0, color='in_tissue_annotation', table_layer='square_002um'
# ).pl.show(
#     coordinate_systems=dataset_id,
#     colorbar=False,
#     dpi=600
# )

In [ ]:
# Total number of spots in table (i.e. spots with at least one UMI)
n_spots = len(sdata.tables['square_002um'].obs)
print(f"Number of spots in table: {n_spots}")

Number of spots in table: 9037460


In [ ]:
# Number of spots in tissue
n_spots_in_tissue = len(sdata.tables['square_002um'][sdata.tables['square_002um'].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue}")

Number of spots in tissue: 8295166


In [ ]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

Capture area covered by tissue: 33.30 mm²


## Capture effeciency metrics

In [ ]:
probe_set_df = pd.read_csv(f"{path_outs}/probe_set.csv", comment='#')
included = set(probe_set_df[probe_set_df['included'] == True]['gene_id'])
adata = sdata.tables['square_002um'][:, sdata.tables['square_002um'].var['gene_ids'].isin(included)]

In [ ]:
# Total number of genes targeted
print(f"Total number of genes targeted: {len(included)}")

Total number of genes targeted: 18085


In [ ]:
# Total number of genes
print(f"Total number of genes: {(adata.X.sum(axis=0) > 0).sum()}")

Total number of genes: 18018


In [ ]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(adata[adata.obs['in_tissue'] == 1].X.sum(axis=0) > 0).sum()}")

Number of genes in tissue: 18016


In [ ]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {adata.X.sum()}")

Total number of unique UMIs: 39873080.0


In [ ]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {adata[adata.obs['in_tissue_annotation'] == 1].X.sum()}")

Number of unique UMIs in tissue: 38927696.0


In [ ]:
import h5py

# Open the HDF5 file
with h5py.File(f"{path_outs}/molecule_info.h5", "r") as f:
    probe_idx = f['/probes/probe_id']
    probe_id = f['/probes/probe_id']
    count = f["/count"]

    total_sum = count[:].sum()
    total_count = count.size

    print("Sum of all numbers:", total_sum)
    print("Total number of entries:", total_count)


Sum of all numbers: 434073772
Total number of entries: 41178835


In [ ]:
import h5py
import numpy as np

# Open the HDF5 file
with h5py.File(f"{path_outs}/molecule_info.h5", "r") as f:
    # Load the feature_id, feature_idx, and count datasets
    feature_id = f['/features/id'][:]
    feature_idx = f['/feature_idx'][:]
    count = f['/count'][:]

    # Decode feature_id entries
    feature_id = np.array([i.decode('ascii') for i in feature_id])
    
    # Find indices to keep
    included_feature_mask = np.isin(feature_id, list(included))
    included_feature_indices = np.where(included_feature_mask)[0]
    included_counts_mask = np.isin(feature_idx, included_feature_indices)
    
    # Sum counts for filtered indices
    included_counts = count[included_counts_mask]
    total_sum = np.sum(included_counts)
    total_count = len(included_counts)

    print("Total number of UMIs for included genes:", total_sum)
    print("Number of unique UMIs for included genes:", total_count)


Total number of UMIs for included genes: 419904640
Number of unique UMIs for included genes: 39873094
